# Assignment 4: DEM, Silo Discharge and the Beverloo Law

**6EMA02 Particle-based Simulations, 2026-2027 edition**
Discrete element method · contacts and friction · granular flow

## How this assignment is assessed: read this first

This notebook is both the assignment text and your report. Task cells
(like this one) are the assignment text and are marked read-only. Jupyter
enforces that; some editors (VS Code among them) do not, so simply leave
them untouched either way: hand-ins are checked against the released
notebook, and modified task cells are flagged. You write your answers in
the cells below each task and hand in this same notebook, **executed**,
together with an HTML export, your code, and the data files the notebook
loads (layout at the end of this notebook).

Your report is **not graded on its content**. Its result is determined in
the individual oral examination at the end of the quartile: the exam is a
conversation about *your* code and *your* results, with this notebook on
screen. Write it as your defence dossier: concise, bullet answers are
fine, every claim something you can stand behind. A report you never hand
in scores 0 for its weight in the final grade; a handed-in report's
result *is* your oral grade.

**AI and sources.** You may use AI assistants (ChatGPT, Claude, Copilot,
…) freely in this course, for code, analysis, and text. Declare the use
briefly in the *Sources* section below, along with any other external
sources: anything in your submission (code, text, data, results) that
your group did not produce itself. Declared sources are never penalised:
nobody grades the text of your report or the style of your code. The oral
examination assesses what *you* understand about the simulation you hand
in. Anything in your submission is fair game, and "we didn't write that
part" is not an available answer. Presenting undeclared external work as
your own is academic misconduct and is treated as such.

## Group

- **Group ID:** …
- **Members:** name (student ID), name (student ID)

## The physics: granular matter is its own state

Sand pours like a liquid, holds a slope like a solid, and sprays like a
gas, sometimes within one second and one apparatus. Two features of the
grains explain almost all of it, and both are absent from A2 and A3.

**Contacts are dissipative.** Two grains that collide do not bounce back
with the same speed: energy goes into deformation, sound and heat. The
loss is quantified by the **coefficient of restitution**
$e = v_\mathrm{after}/v_\mathrm{before}$, which is 0.92 for our
particles: nearly elastic, and still enough that an unstirred granular
system comes to rest. There is no temperature worth speaking of, because
thermal energy is utterly negligible against $m g d_p$ for a millimetre
grain. Without external driving, granular matter simply stops.

**Contacts are frictional, and they remember.** A grain resting on a
slope is held by static friction. Friction is not a function of the
current positions alone: it depends on how much the contact has been
loaded *since it formed*. That is why the code carries a **tangential
displacement per contact** and why contact history has to survive from
step to step. It is the deepest structural difference from your MD code,
where a pair interaction depends only on the current separation.

**How the forces are modelled.** There is no interaction potential here.
Two particles interact only when they geometrically overlap, and the
model is a **spring and dashpot** in the normal direction, plus a
tangential spring limited by the Coulomb condition
$|\mathbf{F}_t| \le \mu |\mathbf{F}_n|$ (the Cundall-Strack model). The
spring gives the elastic rebound, the dashpot the dissipation, and the
Coulomb limit turns sticking into sliding. Two consequences you will meet
directly:

- The spring and dashpot constants are not free parameters you set. They
  are **derived** from a restitution coefficient and a contact time, and
  Part A verifies that the model gives back the restitution it was built
  from.
- The contact time sets the time step: you need many steps per collision
  to resolve it, so $\Delta t$ is tied to the contact duration and not to
  anything about the flow.

**The physics you will actually test.** A silo is a beautiful case
because it violates the intuition trained on liquids. A draining water
tank slows down as it empties, since the pressure at the outlet falls
with the head of liquid. A silo does not: its discharge rate is
**independent of the fill height**. The reason is that grains transmit
stress through force chains that push outward on the walls, so wall
friction carries most of the weight and the pressure near the bottom
saturates once the filling is more than about a diameter deep. This is
Janssen's classic result, and it is why the outflow depends on the
orifice and not on how much is above it.

What is left is a dimensional argument. If the flow is set by grains
falling freely from an arch of the order of the orifice size, the
velocity scales as $\sqrt{gD}$ and the area as $D^2$, so the mass flow
goes as $\rho_b \sqrt{g} D^{5/2}$. Grains cannot pass within one radius
of the rim, which turns $D$ into an effective $D - k d_p$. That is the
**Beverloo correlation**

$$Q = C\,\rho_b\,\sqrt{g}\,(D - k\,d_p)^{5/2},$$

with $Q$ the mass discharge rate, $\rho_b$ the bulk density of the
packed bed, $d_p$ the particle diameter, and $C$ and $k$ empirical
constants: typically $C \approx 0.58$ (reported values 0.55 to 0.65)
and $k \approx 1.5$. Your job in Part D is to test it quantitatively on
your own simulated silo. At small orifices the same physics turns
against the flow: an arch that forms across the opening can be strong
enough to hold, and the silo clogs. That is task D4.

## The model: particles, walls and parameters

| | Normal restitution | Tangential restitution | Friction |
|---|---|---|---|
| Particle-particle | 0.92 | 0.33 | 0.50 |
| Particle-wall (all walls) | 0.85 | 0.33 | 0.45 |

- Particle diameter $d_p = 2.0$ mm (radii uniform between 0.975 and
  1.024 mm), material density $\rho_p = 2500\ \mathrm{kg/m^3}$,
  mean particle mass $1.046 \times 10^{-5}$ kg.
- Contact time $t_c = 0.219$ ms, the same for a particle-particle and a
  particle-wall contact; time step $\Delta t = t_c/20 = 1.095 \times
  10^{-5}$ s.
- The silo, for Parts C and D: 7500 particles in a cylinder of radius
  $R_\mathrm{cyl} = 17.7$ mm (a diameter of 17.7 $d_p$), bottom plate at
  $z_\mathrm{silo} = 53.1$ mm above the floor of the box, settled fill
  about 58 mm high. All of this is already set in
  `set_silo_geometry`: you do not size anything yourself.

## The code, and how to work with it

The starting code is `DEM_student_v7.1.0.zip` under Files ▸ Code. It has
the same structure as your MD code, so most of it will be familiar.
Unpack it into `code/` in your working directory.

**Every piece of code you write already exists** as a function that
compiles, runs, and is called from the right place; the `\todo` inside
it says exactly what to fill in. Open `html/index.html` from the zip: its
*todo list* has them all. In short:

| Part | function, file | what you write |
|---|---|---|
| A | `initialise_collision`, initialise.c | one particle at the wall, or two head-on |
| B | `update_parameters`, setparameters.c | rotate gravity from $t_\mathrm{tilt}$ on |
| C1 | `cylinder_wall`, walls.c | the silo wall |
| C2 | `silo_bottom_wall`, walls.c; `update_parameters` | the edge of the hole in the plate (the flat plate is given), and opening the orifice at $t_\mathrm{open}$ |
| C3 | `initialise_positions_silo`, initialise.c | the fill: a loose lattice inside the cylinder |
| D2 | `count_in_silo`, walls.c | the number of particles above the plate |

That is all the C you write. **All analysis is done in Python**, from the
table the program prints and from the trajectory file.

**Choosing a run.** The macro `RUN` at the top of `setparameters.c`
selects the set-up: 1 is the test collision of Part A, 2 the tilted bed
of Part B, 3 the silo of Parts C and D. It and the other settings you
vary (`NUM_TEST`, `V_IMPACT`, `DT_FRACTION`, `TILT_RATE`, `D_ORIFICE`,
`T_DISCHARGE`) have defaults there and can be given on the command line,
so every run is one command you can copy into your build log:

```
gcc -O3 -DRUN=1 -DNUM_TEST=2 -DV_IMPACT=1.0 *.c -o dem -lm    # Part A, two particles head-on
gcc -O3 -DRUN=2 *.c -o dem -lm                                # Part B
gcc -O3 -DRUN=3 -DD_ORIFICE=8 -DT_DISCHARGE=1.0 *.c -o dem -lm  # Part D, orifice of 8 d_p
./dem > run.dat
```

(Linux and macOS; on Windows use MSYS2. For debugging:
`gcc -g -Wall -Wextra *.c -o dem_debug -lm`.)

**Output.** Every run prints a table: step, time, coordination number
$Z$ (contacts per particle), potential, kinetic and total energy, and
`N_silo`, the number of particles in the silo. The first line, starting
with `# run`, records the geometry and the time step. Redirect the table
to a file and load it with `numpy.loadtxt`, which skips the `#` lines.
Every run also writes `trajectories.xyz`, a frame every 0.022 s.

**Each run in its own directory.** Every run writes `trajectories.xyz`
and `restart.dat` into the directory it runs in, so two runs in one
directory overwrite each other. Make a directory per run, e.g.
`mkdir -p runs/D8 && cd runs/D8 && ../../code/dem > run.dat`. Runs in
different directories can go in parallel on different cores.

**Visualisation.** Open `trajectories.xyz` in OVITO: the file states the
box, the radii and the speed itself, so no column mapping is needed.
Colour by speed (*Add modification ▸ Color coding ▸* `speed`), and to see
inside the silo add *Slice* (normal (0, 1, 0), slab width 4 mm, through
the centre of the box). Without the slice you only see the particles
against the wall.

**Reading a trajectory in Python.** A frame is a line with the number of
particles, a line with the box and the time, then one line per particle:
`C x y z radius speed`, in SI units. This reads all frames:

```python
import numpy as np, re
def read_xyz(path):
    frames, times = [], []
    with open(path) as f:
        while (line := f.readline()):
            n = int(line)
            times.append(float(re.search(r"Time=(\S+)", f.readline()).group(1)))
            rows = [f.readline() for _ in range(n)]
            frames.append(np.loadtxt(rows, usecols=(1, 2, 3, 4, 5)))
    return np.array(times), frames      # each frame: columns x, y, z, radius, speed
```

Do not hand in trajectories (a silo run writes 20 to 60 MB). Save the one
frame you analyse, e.g. with `np.savetxt`, and the images you make.

**What you do not need to do.** No restart files: every run settles from
the same random seed, so the bed at $t_\mathrm{tilt}$ or
$t_\mathrm{open}$ is identical in every run of a kind. No analysis code
in C. No sizing of the silo. No hunting for the right run length: the
table below gives it.

**Python environment:** the same `pbs-env` as in the earlier
assignments, from `requirements.txt`; `nb2html.py` for the HTML you hand
in and `check_my_submission.py` before you submit. See the Canvas page
*Working on the assignments*.

## The runs, at a glance

Times are for one core of a reference workstation; a laptop may take two
or three times longer. Apart from the 24 tiny runs of Part A, seven runs in
all, about 15 minutes of computing on the reference machine.

| task | command (`gcc -O3 … *.c -o dem -lm`) | simulated | CPU |
|---|---|---|---|
| A | `-DRUN=1 -DNUM_TEST=1` or `2`, `-DV_IMPACT=0.1`, `0.3`, `1.0`, `-DDT_FRACTION=0.2`, `0.1`, `0.05`, `0.025`: 24 runs, a shell loop | 2 ms | < 1 s each |
| B | `-DRUN=2` (20 °/s) and `-DRUN=2 -DTILT_RATE=10` | 1.5 s, 2.7 s | 25 s, 45 s |
| C, D1, D2, D3 | `-DRUN=3` ($D = 7 d_p$, the default) | 2.3 s | 4 min |
| D3 | `-DRUN=3 -DD_ORIFICE=6 -DT_DISCHARGE=1.5`, `=8 … =1.0`, `=9 … =0.8` | 1.1 to 1.8 s | 2 to 3 min each |
| D4 | `-DRUN=3 -DD_ORIFICE=3 -DT_DISCHARGE=0.8` | 1.1 s | 2 min |

Long runs are done in the terminal, never from the notebook; let the
notebook load their output from `data/`.

## Code map: fill in as you go

The examiner uses this table in the oral to jump straight to your code,
so **make the entries clickable links**: a markdown link with a path
relative to this notebook, for example

```
| C1 cylinder wall | [`cylinder_wall`](code/walls.c) | [D7.dat](data/D7.dat) |
```

Relative paths only (`code/…`, `data/…`); everything you link to must be
inside the zip you hand in. `check_my_submission.py` verifies this.

| Task | File / function(s) | Data files |
|------|--------------------|------------|
| A test collision | … | … |
| B rotating gravity | … | … |
| C1 cylinder wall | … | … |
| C2 orifice wall | … | … |
| C3 fill | … | … |
| D2 particles in the silo | … | … |

## Sources

**AI tools used:** … / none.
**Used for:** … .
**Other external sources** (code, text, data, or results not produced by
us): … / none.
**Parts of the submission with substantial external content:** … .
**What we did ourselves to verify these parts:** … .

## Build & run log

Exact compile and run commands, machine, and approximate wall-clock time
for each production run whose output sits in `data/`:

- …

## Part A: Verification, restitution

*In the oral you should be able to:* explain where the spring and dashpot
constants come from and why the time step is tied to the contact time;
explain why recovering the restitution coefficient closes the loop on the
contact model; explain why wall and particle-particle collisions need
different constants.

This is verification in its purest form: the code derives its constants
from a number, so the simulation must give that number back. If it does
not, nothing later in this assignment can be trusted.

Write `initialise_collision` (`RUN=1`). Gravity is off in this run and
the impact is head-on, so the particles do not spin and all the energy
is translational: then $e^2 = E_\mathrm{kin,after}/E_\mathrm{kin,before}$,
and both are in the table, which this run prints every time step. No
other output is needed.

### A1) Particle-wall collision

One particle onto the bottom wall (`NUM_TEST=1`), against the
tabulated 0.85.

### A2) Two-particle collision

Two particles head-on (`NUM_TEST=2`), against the tabulated 0.92.

Deliver for both:

- a **figure** of $E_\mathrm{kin}$ and $E_\mathrm{pot}$ against time
  through one collision (0.3 m/s), with the contact marked;
- a **table** of the measured $e$ at impact speeds 0.1, 0.3 and 1.0 m/s
  and at $\Delta t / t_c$ = 0.2, 0.1, 0.05 and 0.025, next to the
  tabulated value. The model is a *linear* spring and dashpot, for which
  the exact $e$ does not depend on the impact speed at all. So a spread of
  $e$ over the three speeds is an error of the time integration, and it is
  the clearest sign of a time step that is too large: one speed alone can
  look right by luck;
- a statement of the **time step dependence**: how many steps per
  contact do you need before $e$ is within 1 % of the prescribed value at
  all three speeds? Is $t_c/20$, the value the code uses, enough? Do not
  expect the error to shrink smoothly with $\Delta t$: look at the
  dashpot force at the moment a contact begins, and say why.

Then explain in two sentences why the particle-wall and
particle-particle cases need different spring and dashpot constants
(`set_parameters`) even though both contacts last $t_c$.

*Results (A1, A2):* measured e against tabulated, dt dependence …

In [ ]:
# A: E_kin and E_pot through one collision; e against impact speed and time step (load from data/)

## Part B: Validation, granular slope stability

*In the oral you should be able to:* explain why rotating gravity is
equivalent to tilting the floor; explain the angle you found from the
way the particles move; explain what static friction requires of the
contact model, and what would happen to this test without the stored
tangential displacement.

The bed of the starting code (2000 particles, about five layers on the
flat bottom wall, periodic sides) settles under vertical gravity until
$t_\mathrm{tilt} = 0.3$ s. Then gravity rotates about the $y$ axis at a
constant rate, which tilts the bed without moving any wall. Write
`update_parameters` (`RUN=2`).

**Before you run, predict:** if the bed slides over the floor as one
block, at which angle does it start to move? (Only one friction
coefficient matters for that.)

Deliver:

- a **figure** of $E_\mathrm{kin}$ (log scale) against slope angle, for
  the two tilt rates of the run table on one plot, with the onset angle
  $\theta_c$ marked. The kinetic energy creeps up before the bed really
  goes: define your onset (for example the angle where $E_\mathrm{kin}$
  passes $10^{-6}$ J, a mean speed of about 1 cm/s) and state the
  definition;
- a **number**: $\theta_c$, and whether it depends on the tilt rate;
- a **comparison** with your prediction.

Do not be alarmed if the bed fails far below your prediction: that is
the point of this task, not an error in your code. Find out why. Take a
frame of the trajectory at a slope of about 20° and compare the mean
speed of the bottom layer (centres less than 2 mm above the floor) with
the mean speed of the particles above it. What ratio would you expect if
the bottom layer rolled, without slipping, under a bed that slides over
it? What does your measured ratio say about how the bed fails, and why is
the friction coefficient then not what sets $\theta_c$?

*Results (B):* …

In [ ]:
# B: E_kin against slope angle for both tilt rates; speed of the bottom layer against the layers above

## Part C: Building the silo

*In the oral you should be able to:* state the wall-function contract and
walk through your cylinder implementation and your edge of the hole,
including why your point is the closest one; explain why the fill needs no separate randomisation; interpret
your packing profile and the bulk density you use for Beverloo.

All of Part C runs inside the one silo run of the run table (`RUN=3`,
$D = 7\,d_p$): the fill settles on the closed plate until
$t_\mathrm{open} = 0.3$ s, after which the orifice opens. Parts C3 and C4
use the part before $t_\mathrm{open}$, Part D the part after.

### C1) Cylindrical wall

Write `cylinder_wall`. The contract (overlap test, $\mathbf{r}_{iw}$
from the closest wall point to the particle centre, $\mathbf{v}_w$) is
documented at the top of `walls.c`; compare with `bottom_wall`.

### C2) Bottom plate with a circular orifice

The flat plate is given in `silo_bottom_wall`; it shows the wall contract
at work. What is missing is the **edge of the hole**: a particle whose
centre is over the open hole, but within one radius of the plane of the
plate, can still touch the plate where the hole begins. Work out which
point of the plate is then closest to the particle centre, and complete
`silo_bottom_wall` with that contact. Open the orifice at
$t_\mathrm{open}$ in `update_parameters`.

Deliver, in the results cell: the **closest wall point** and
$\mathbf{r}_{iw}$ for a centre over the hole, as a formula, with a
sketch or one sentence that shows why it is the closest point; and how
you convinced yourself that your implementation is right. Without the
edge the code still runs and the silo still empties: say what a particle
at the edge of the hole would do then.

### C3) Filling and settling

Write `initialise_positions_silo`: a loose lattice clipped to the
cylinder, as the `\todo` describes. Gravity and the closed plate do the
rest.

Deliver a **figure** of $E_\mathrm{kin}$ and the coordination number
$Z$ against time for $t < t_\mathrm{open}$, showing that the fill has
come to rest before the orifice opens, and a **snapshot** of the settled
bed (OVITO, sliced).

### C4) The packing

From the last trajectory frame before $t_\mathrm{open}$, deliver:

- a **number**: the bulk solids volume fraction $\phi$ and the bulk
  density $\rho_b = \phi \rho_p$, which you need for Beverloo. Use a core
  region away from the walls: centres closer than $R_\mathrm{cyl} - 2d_p$
  to the axis, and between $2d_p$ above the plate and $3d_p$ below the
  highest particle. $\phi$ is the summed volume of the particles whose
  centre is in the core, divided by the core volume;
- a **number**: the coordination number $Z$ of the settled packing (the
  table has it);
- a **figure** of the radial profile of the particle centres: the number
  of centres per volume in thin cylindrical shells (0.2 mm) over the core
  height, times the mean particle volume. Near the wall this is *not* a
  local volume fraction, and it can exceed 0.74: the centres of a layer
  of spheres against a wall all sit at one distance from it.

Then discuss: why does the profile oscillate near the wall, and over how
many particle diameters? What does that imply for how wide a silo has to
be, compared with its orifice?

*Results (C):* …

In [ ]:
# C3/C4: settling (E_kin and Z against time); bulk phi and rho_b in the core; radial profile

## Part D: Discharge and the Beverloo law

*In the oral you should be able to:* explain why the discharge rate does
not depend on the fill height (and why a liquid's does); derive which
quantities the Beverloo scaling can be built from dimensionally; explain
what happens at small orifices and why.

Write `count_in_silo`: then the last column of the table is $N(t)$, the
number of particles still in the silo. Time zero of the discharge is
$t_\mathrm{open} = 0.3$ s.

### D1) Discharge

From the silo run at $D = 7\,d_p$, deliver **snapshots** at three times
(early, middle, late), sliced through the axis and coloured by speed,
and mark on them where the material flows and where it is stagnant.
Describe the shape of the flowing region and say what it means for how a
silo empties: which grains leave first, those at the top or those at the
bottom?

### D2) Discharge rate

From the same run, extract the steady discharge rate $Q$, in particles
per second and in g/s (mean particle mass above).

Deliver:

- a **figure** of $N(t)$ with the fitted window marked. Fit from the
  moment 10 % of the particles has left until 60 % has left (or the run
  ends): the start-up and the last, shallow part of the fill are not
  steady;
- a **figure** of the rate against the number of particles left (a
  running slope over about 0.1 s), which is where the constancy is
  actually visible;
- a **number**: $Q$ with an error estimate, and evidence that it is
  constant while the fill height drops. Fit the first and the second half
  of the window separately and compare: that is the signature which
  distinguishes a silo from a water clock.

Then explain the physics: why does the head of material above the orifice
not matter, and why does it matter for a draining water tank?

### D3) Beverloo scaling

Add orifices of 6, 8 and 9 $d_p$ (run table) to your 7 $d_p$.

Deliver:

- a **table** of $D$, $D/d_p$, $Q$ and its error, per run;
- a **figure** of $Q^{2/5}$ against $D$ with a linear fit, since Beverloo
  predicts a straight line whose intercept gives $k\,d_p$;
- **numbers**: your fitted $k$ and $C$ with uncertainties, using your
  $\rho_b$ from C4, next to the literature values, and a quantitative
  statement about the quality of the fit;
- a sentence on which of your fitted constants you trust more, and why.

Then derive, dimensionally, why the exponent is $5/2$ for a
three-dimensional silo, and state what it would be in two dimensions.

Finally, the cost of doing better. At $D = 9\,d_p$ the silo is only two
orifices wide, while Beverloo assumes a silo much wider than its
orifice. Estimate, from your own run times, what the four D3 runs would
cost in a silo twice as wide with the same fill height. Say which
factors of that estimate scale with the number of particles and which
do not, and why the time step does not change.

### D4) The small-orifice limit

Run $D = 3\,d_p$.

Deliver a **figure** of $N(t)$ for this run next to your $7\,d_p$ run on
the same axes, a **snapshot** of the blocked orifice, and a description
of what you observe. Explain the mechanism qualitatively, and say what
this implies for the range of validity of the Beverloo correlation you
fitted in D3.

*Results (D):* Q(t) …, Beverloo fit k = …, C = …, small-orifice
observations …

In [ ]:
# D: snapshots; N(t) and Q with the half-window check; Q^(2/5) against D with linear fit; the small orifice

## Lessons learned

What would you do differently if you started over? What surprised you?
What in your results still bothers you? This is the last assignment of
the course: also say which of the four methods you would reach for first
if a new problem landed on your desk, and why.

*Lessons learned:* …

## Hand-in

One zip per group, layout exactly:

```
<group>_A4.zip
├── A4.ipynb     this notebook, filled in and executed
│                (all outputs present; task cells untouched)
├── A4.html      python3 nb2html.py A4.ipynb
├── code/        all .c/.h + README.md; must build with
│                gcc -O3 *.c -o dem -lm    (no binaries)
└── data/        every data file and image this notebook loads
                 (no trajectories)
```

Long runs are done in the terminal; record the exact commands in the
*Build & run log* and let the notebook load the saved output from
`data/`. Handed in executed, the notebook must open and display without
rerunning anything.

**Before you submit,** check your own zip:

```
python3 check_my_submission.py <group>_A4.zip --release A4.ipynb
```

where `--release` is a freshly downloaded, unmodified `A4.ipynb`. It
verifies the layout, that the task cells are untouched, that every cell
has been executed, that your C code compiles, and that every path and
link in the notebook is relative and present in the zip. Fix what it
reports, then submit. Setting up the Python environment
(`requirements.txt`, both helper scripts) is described on the Canvas page
*Working on the assignments*.

All four notebooks must be handed in before the oral examinations start.